# 🚀 JAX: transformations of functions

*Section 15.3 of the notes.*

JAX **transforms functions**. `grad`, `vmap`, and `jit` each take a function and
return a new function, and they can be combined in any order:

* `grad(f)`: the derivative of $f$, by a reverse sweep;
* `vmap(f)`: $f$ applied to a whole batch, without a Python loop;
* `jit(f)`: $f$ compiled, so that it runs without the interpreter.

All three work by **tracing**: JAX runs the function once with placeholders
instead of numbers, records the operations, and keeps the graph. The price is
that the function must be **pure** (previous notebook) and traceable: no
hidden state, no arrays changed in place, no `if` on the value of an argument.

This notebook shows the transformations, the rules that tracing imposes, and
what compiling costs and saves.

## ⚙️ Setup

JAX computes in single precision by default; the line `jax_enable_x64`
switches to double precision. `jax.numpy`, imported as `jnp`, has the functions
of NumPy for JAX arrays.

JAX runs **asynchronously**: a call returns before the result is computed.
When we time, `.block_until_ready()` waits for the result, so that we measure
the computation and not only its start. `time.perf_counter()` reads a clock in
seconds; the difference of two readings is the time between them.

In [ ]:
import time

import jax
import jax.numpy as jnp
import numpy as np

jax.config.update("jax_enable_x64", True)
print("jax", jax.__version__, "| device:", jax.devices()[0])

## 1. 🔧 Three transformations

Take $f(x) = \sin(x^2)$ again, with $f'(1) \approx 1.0806$.

In [ ]:
def f(x):
    return jnp.sin(x**2)


df = jax.grad(f)  # x -> f'(x)
d2f = jax.grad(jax.grad(f))  # x -> f''(x)
df_batch = jax.vmap(df)  # f' at every entry of an array
df_jit = jax.jit(df)  # f', compiled

print("f'(1)          ", df(1.0))
print("f''(1)         ", d2f(1.0))
print("f' at 1, 2, 3  ", df_batch(jnp.array([1.0, 2.0, 3.0])))
print("compiled f'(1) ", df_jit(1.0))

💡 The same numbers as `torch.func` in notebook 02. `jax.jit(jax.grad(f))`
compiles the reverse sweep itself.

## 2. 🧊 Arrays cannot be changed

A trace must be a fixed sequence of operations, so JAX arrays are
**immutable**.

🤔 *Guess: what happens with `x[0] = 5.0`?*

In [ ]:
x = jnp.arange(4.0)
try:
    x[0] = 5.0
except TypeError as err:
    print("TypeError:", err)

y = x.at[0].set(5.0)  # a new array with one entry changed
print("x:", x)
print("y:", y)

💡 `x.at[0].set(5.0)` is written as if it made a copy, and `x` is unchanged.
Inside `jit`, when `x` is no longer needed, the compiler turns it into a write
in place, so the copy costs nothing.

## 3. 🎲 Randomness is explicit

NumPy keeps a hidden random generator whose state changes with every draw. A
hidden state breaks purity, so JAX has none: you pass a **key**, and you
**split** it to get new ones.

* `jax.random.key(0)` makes a key from the number 0;
* `jax.random.split(key)` returns two new keys;
* `jax.random.normal(key, (3,))` draws 3 normal numbers from a key.

🤔 *Guess: the same key twice gives...?*

In [ ]:
key = jax.random.key(0)
k1, k2 = jax.random.split(key)
first = jax.random.normal(k1, (3,))
again = jax.random.normal(k1, (3,))
other = jax.random.normal(k2, (3,))
print("k1:      ", first)
print("k1 again:", again)
print("k2:      ", other)

💡 The same key always gives the same numbers, whatever ran before. A function
that draws random numbers is therefore a function of its key, and a run
repeats exactly, on any machine and in any order.

## 4. 🛠️ Transforming the Euler solver

The Euler solver of notebook 03 is plain Python and knows nothing about JAX.
We apply the three transformations to it **without changing a line**. The
transformations need a function of the array only, so `run` fixes the time
step, the number of steps, and the vector field.

In [ ]:
def euler_step(u, dt, f):
    return u + dt * f(u)


def euler_run(u0, dt, steps, f):
    u = u0
    for _ in range(steps):
        u = euler_step(u, dt, f)
    return u


def decay(u):
    return -u


def run(u0):
    # 100 steps of size 0.01 for du/dt = -u: a function of the initial state only.
    return euler_run(u0, 0.01, 100, decay)


finals = jax.vmap(run)(jnp.array([1.0, 2.0, 3.0]))  # three initial states
derivative = jax.grad(run)(1.0)  # d u_100 / d u_0 at u_0 = 1
final_jit = jax.jit(run)(1.0)

print("vmap: final states  ", finals)
print("grad: d u_100 / d u0", derivative)
print("formula (1 - dt)^100", (1 - 0.01) ** 100)
print("jit:  final state   ", final_jit)

💡 `grad` differentiated the final state with respect to the initial state
**through all 100 steps**: each step multiplies by $1 - \Delta t$, so the
derivative is $(1 - \Delta t)^{100}$, as printed.

## 5. 🔁 A Python loop or `scan`?

To compile `euler_run`, `jit` first **traces** it: it runs the function once,
with placeholders in place of numbers, and records every operation it meets.
The Python `for` loop inside `euler_run` runs too, 1000 times, and each pass
records the operations of one step. The recorded graph therefore contains
**1000 copies** of the step. `jax.lax.scan` takes the body of the loop as a function and the
number of steps as a number, so the graph contains the body **once**.

`jax.lax.scan(body, u0, None, length=steps)` runs the loop

```python
u = u0
for i in range(steps):
    u, _ = body(u, None)
```

and returns `u` and the list of the second outputs of `body`. Our body has no
second output, so it returns `None` there, and we ignore that list with `_`.
The third argument, `None`, says that the loop runs over no array; `length`
gives the number of steps.

We compile both versions for 1000 steps of size $0.001$ on a state of 2048
numbers, and time the first call and the later calls.

🤔 *Guess: how much longer does the first call (trace and compile) take for
the Python loop?*

In [ ]:
def euler_scan(u0, dt, steps, f):
    def body(u, _):
        return euler_step(u, dt, f), None

    u, _ = jax.lax.scan(body, u0, None, length=steps)
    return u


loop_jit = jax.jit(lambda u: euler_run(u, 0.001, 1000, decay))
scan_jit = jax.jit(lambda u: euler_scan(u, 0.001, 1000, decay))
u0 = jnp.ones(2048)

t0 = time.perf_counter()
loop_jit(u0).block_until_ready()
t_first_loop = time.perf_counter() - t0

t0 = time.perf_counter()
scan_jit(u0).block_until_ready()
t_first_scan = time.perf_counter() - t0

t_later_loop = %timeit -o -q loop_jit(u0).block_until_ready()
t_later_scan = %timeit -o -q scan_jit(u0).block_until_ready()

print(
    f"Python loop: first call {1e3 * t_first_loop:5.0f} ms   later calls {1e6 * t_later_loop.best:5.0f} us"
)
print(
    f"scan:        first call {1e3 * t_first_scan:5.0f} ms   later calls {1e6 * t_later_scan.best:5.0f} us"
)

💡 **What happened?** Compiling 1000 copies of the step takes far longer than
compiling one, and the compiled code is much larger. The later calls cost
about the same.

👉 Write time loops inside `jit` with `scan` (or `jax.lax.fori_loop`).

## 6. ⏱️ What `jit` costs and what it saves

The first call of a compiled function **traces and compiles**; later calls run
the compiled code. The example is the one quoted in Section 15.3: twenty
updates of a $64\times64$ matrix.

🤔 *Guess: how long does compiling take, and how much faster is the compiled
version?*

In [ ]:
def update(w, x, steps):
    for _ in range(steps):
        w = w - 0.01 * jnp.tanh(w @ x) @ x.T
    return w


key = jax.random.key(1)
kw, kx = jax.random.split(key)
w = jax.random.normal(kw, (64, 64))
x = jax.random.normal(kx, (64, 64))

update_jit = jax.jit(
    update, static_argnums=2
)  # steps is a number, fixed at compile time
t0 = time.perf_counter()
update_jit(w, x, 20).block_until_ready()
t_first = time.perf_counter() - t0
print(f"first call (traces and compiles): {1e3 * t_first:.0f} ms")

t_plain = %timeit -o -q update(w, x, 20).block_until_ready()
t_jit = %timeit -o -q update_jit(w, x, 20).block_until_ready()
ratio = t_plain.best / t_jit.best
print(f"without jit {1e6 * t_plain.best:7.0f} us")
print(f"with jit    {1e6 * t_jit.best:7.0f} us   ratio {ratio:.1f}x")

💡 **What happened?** Compiling takes about a tenth of a second, the cost of
about two hundred calls of the compiled version. The compiled version runs without the interpreter between
the operations and lets the compiler fuse them.

`static_argnums=2` tells `jit` that `steps` is not an array but a number that
fixes the shape of the computation. A call with a **new** value of `steps`, or
with arrays of a new shape or type, compiles again.

## 7. 🚦 Branching on a value

A trace must be the same for every value of the arguments. A Python `if` on the
value of an argument cannot be traced.

🤔 *Guess: does `jax.jit(piecewise)` work?*

In [ ]:
def piecewise(x):
    if x > 0:  # depends on the value of x
        return x**2
    return -x


try:
    jax.jit(piecewise)(1.0)
except jax.errors.TracerBoolConversionError as err:
    print("TracerBoolConversionError:", err)

💡 **What happened?** During tracing `x` is a placeholder, so `x > 0` has no
value, and Python cannot decide the `if`.

`jax.lax.cond(condition, f_true, f_false, x)` returns `f_true(x)` if the
condition holds and `f_false(x)` otherwise. It puts **both** branches into the
graph and chooses one when the function runs.

In [ ]:
def piecewise_traced(x):
    return jax.lax.cond(x > 0, lambda v: v**2, lambda v: -v, x)


piecewise_jit = jax.jit(piecewise_traced)
print("at  1.0:", piecewise_jit(1.0))
print("at -2.0:", piecewise_jit(-2.0))

A placeholder has no values, but it has a **shape**: `jit` compiles a separate
version for each shape of the arguments, and knows that shape while it traces.
An `if` on the shape can therefore be decided during tracing. The function
below squares a single number and sums the squares of a vector.

In [ ]:
def square_or_sum(x):
    if x.ndim == 0:  # depends only on the shape of x
        return x**2
    return jnp.sum(x**2)


square_or_sum_jit = jax.jit(square_or_sum)
print("a number:", square_or_sum_jit(3.0))
print("a vector:", square_or_sum_jit(jnp.array([1.0, 2.0, 3.0])))

## 8. 🌡️ Allen–Cahn: NumPy against JAX

Back to the race of notebook `04-numba` of Chapter 14: 2000 forward Euler
steps of the Allen–Cahn equation on the periodic grid of Chapter 2, with
$d = 2048$ points. The NumPy version runs the time loop in Python; the JAX
version compiles the whole loop with `scan`.

* `@jax.jit` above a function definition is the same as writing
  `ac_march_jax = jax.jit(ac_march_jax)` after it.
* `jnp.asarray` turns a NumPy array into a JAX array, and `np.asarray` turns a
  JAX array back into a NumPy array.
* `np.allclose(a, b)` checks that two arrays agree up to rounding.

🤔 *Guess the ratio!*

In [ ]:
def ac_step_numpy(u, dt, dx, diffusion):
    lap = (np.roll(u, -1) - 2 * u + np.roll(u, 1)) / dx**2
    return u + dt * (diffusion * lap + u - u**3)


def ac_march_numpy(u, dt, dx, diffusion, steps):
    for _ in range(steps):
        u = ac_step_numpy(u, dt, dx, diffusion)
    return u


def ac_step_jax(u, dt, dx, diffusion):
    lap = (jnp.roll(u, -1) - 2 * u + jnp.roll(u, 1)) / dx**2
    return u + dt * (diffusion * lap + u - u**3)


@jax.jit
def ac_march_jax(u, dt, dx, diffusion):
    def body(v, _):
        return ac_step_jax(v, dt, dx, diffusion), None

    v, _ = jax.lax.scan(body, u, None, length=2000)
    return v


d = 2048
grid = -1 + 2 * np.arange(d) / d
u0 = grid**2 * np.cos(np.pi * grid)
dx, dt, diffusion = 2 / d, 0.005, 2e-5

u0_jax = jnp.asarray(u0)
u_numpy = ac_march_numpy(u0, dt, dx, diffusion, 2000)
u_jax = ac_march_jax(u0_jax, dt, dx, diffusion)  # the first call compiles
print("same result:", np.allclose(u_numpy, np.asarray(u_jax)))

t_np = %timeit -o -q ac_march_numpy(u0, dt, dx, diffusion, 2000)
t_jx = %timeit -o -q ac_march_jax(u0_jax, dt, dx, diffusion).block_until_ready()
ratio = t_np.best / t_jx.best
print(f"numpy {1e3 * t_np.best:7.1f} ms")
print(f"jax   {1e3 * t_jx.best:7.1f} ms   ratio {ratio:.1f}x")

💡 **What happened?** As with numba, the compiled time loop pays no
interpreter cost per step and allocates no temporaries. Unlike numba, the JAX
version can also be differentiated with `grad` and batched with `vmap`, since
it is the same pure function.

## 🎯 Summary

| Rule of tracing | Why | Instead |
|---|---|---|
| no hidden state | traced once, then replayed | pass everything as arguments |
| arrays are immutable | the trace is a fixed sequence of operations | `x.at[i].set(v)` |
| no `if` on a value | the trace must fit every value | `jax.lax.cond` |
| loops are unrolled | 1000 steps, 1000 copies | `jax.lax.scan` |
| randomness is explicit | a hidden generator is hidden state | keys and `jax.random.split` |

The first call of `jit` compiles; a new shape, type, or static argument
compiles again.

## 🧪 Try it yourself

1. In Section 4, compute `jax.grad(run)` at many initial states at once by
   combining it with `vmap`. Why are all the values the same?
2. In Section 6, call `update_jit(w, x, 21)`. Time the first call. What
   happened?
3. In Section 8, change $d$ to $2^{16}$ and `dt` to `1e-6`. Does the ratio
   between NumPy and JAX grow or shrink? Compare with the numba notebook.